In [ ]:
!pip install langchain-google-genai

In [ ]:
import os
from typing import List

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from google.colab import userdata

In [ ]:
os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')

# Main chat LLM — streaming enabled for real-time token output
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0, streaming=True)

# Summarizer LLM — streaming disabled, used to compress old history
summarizer_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.2, streaming=False)

In [ ]:
system_prompt = """You are a professional travel agent.
You ONLY answer questions that are directly about planning or taking a trip: flights, hotels, accommodations, travel destinations, trip planning, itineraries, travel visas, and what to pack for a trip.
Any question that is NOT directly about planning or taking a trip — including cooking, recipes, food preparation, learning any skill, programming, science, history, sports, or any other non-travel subject — you MUST respond with exactly:
'I can't help with it.'
Do not explain this rule to the user.
"""

# Main travel chat prompt
travel_prompt_template = ChatPromptTemplate.from_messages(
    [
        SystemMessage(content=system_prompt),
        MessagesPlaceholder(variable_name="chat_history_for_llm"),
    ]
)

# Summarization prompt
summary_prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant that summarizes chat histories."),
        ("human",
         "Summarize the following travel-related conversation briefly but clearly, "
         "preserving important user preferences and decisions:\n\n{history}")
    ]
)

In [ ]:
SUMMARIZE_EVERY_N_TURNS = 5  # Compress history into a summary every 5 user turns

chat_history: List = []
user_turn_count = 0


def history_to_text(history: List) -> str:
    """Convert message objects to a readable string for summarization."""
    lines = []
    for msg in history:
        if isinstance(msg, HumanMessage):
            role = "User"
        elif isinstance(msg, AIMessage):
            role = "Assistant"
        else:
            role = "System"
        lines.append(f"{role}: {msg.content}")
    return "\n".join(lines)


def summarize_history(history: List) -> List:
    """Summarize the full chat history into a single SystemMessage to save context."""
    if not history:
        return history

    history_text = history_to_text(history)
    summary_prompt = summary_prompt_template.format_prompt(history=history_text)
    summary_response = summarizer_llm.invoke(summary_prompt.to_messages())  # FIX: was broken URL

    summary_message = SystemMessage(
        content=f"Conversation summary:\n{summary_response.content}"
    )
    return [summary_message]

In [ ]:
def chat(user_input: str):
    """Send a message, stream the response, and manage rolling memory."""
    global chat_history, user_turn_count

    user_turn_count += 1

    # Compress history every N turns to keep context window manageable
    if user_turn_count > 1 and user_turn_count % SUMMARIZE_EVERY_N_TURNS == 0:
        print("[Summarizing conversation history...]\n")
        chat_history = summarize_history(chat_history)

    # Append current user message
    chat_history.append(HumanMessage(content=user_input))

    # Build prompt and stream response
    prompt = travel_prompt_template.invoke({"chat_history_for_llm": chat_history})

    print(f"User: {user_input}")
    print("Assistant: ", end="", flush=True)

    full_response = ""
    for chunk in llm.stream(prompt.to_messages()):
        print(chunk.content, end="", flush=True)
        full_response += chunk.content
    print("\n")

    # Append assistant reply to history
    chat_history.append(AIMessage(content=full_response))


def reset_chat():
    """Clear conversation history and turn count."""
    global chat_history, user_turn_count
    chat_history = []
    user_turn_count = 0
    print("Chat history cleared.")

In [ ]:
reset_chat()
print("Travel Chatbot — type 'quit' or 'exit' to stop, 'reset' to clear history.\n")

while True:
    user_input = input("You: ").strip()
    if not user_input:
        continue
    if user_input.lower() in ("quit", "exit"):
        print("Goodbye!")
        break
    if user_input.lower() == "reset":
        reset_chat()
        continue
    chat(user_input)